In [3]:
import numpy as np

##3 words, each a tiny 2-number vector:[ about money?, direction(+up,- down)]
words = ["profit", "fell", "sharpely"]
X = np.array([[1.0,0.0],  #profit
              [0.2,-1.0],  ##fell
              [0.0,-0.8]])   ## sharply

##step 1: how well does "fell" match each word? (dot product = similarity)
scores = X[1] @ X.T
print("scores :", scores.round(2))

##step 2 : turn scores into weights that add up to 1 (softmax)
weights = np.exp(scores) / np.exp(scores).sum()
print("weights:", weights.round(2),"-> sum=", weights.sum().round(2))

#step 3: new meaning of "fell" = weighted average of all words
new_fell = weights @ X
print("old fell:", X[1], " new fell:", new_fell.round(2))

scores : [0.2  1.04 0.8 ]
weights: [0.19 0.45 0.35] -> sum= 1.0
old fell: [ 0.2 -1. ]  new fell: [ 0.28 -0.73]


In [4]:
Wq = np.array([[0.0, 0.0],
               [-1.0, 0.0]])   # turns "fell" into the question: "who is about money?"
Wk = np.eye(2)                  # keys = the words as they are (identity matrix)
Wv = np.eye(2)                  # values = the words as they are

Q = X @ Wq
K = X @ Wk
V = X @ Wv

scores = Q[1] @ K.T             # fell's QUESTION vs every word's LABEL
weights = np.exp(scores) / np.exp(scores).sum()
new_fell = weights @ V

print("fell's query:", Q[1])
print("scores      :", scores.round(2))
print("weights     :", weights.round(2))
print("new fell    :", new_fell.round(2))

fell's query: [1. 0.]
scores      : [1.  0.2 0. ]
weights     : [0.55 0.25 0.2 ]
new fell    : [ 0.6  -0.41]


In [5]:
rng = np.random.default_rng(42)
softmax = lambda s: np.exp(s - s.max()) / np.exp(s - s.max()).sum()

for d in [2, 64, 768]:
    q = rng.normal(size=d)            # one query
    K = rng.normal(size=(5, d))       # keys of 5 words
    s = K @ q                         # raw scores
    print(f"d={d:4d} | score spread={s.std():6.2f} | "
          f"top weight raw={softmax(s).max():.2f} | scaled={softmax(s / np.sqrt(d)).max():.2f}")

d=   2 | score spread=  0.67 | top weight raw=0.34 | scaled=0.30
d=  64 | score spread=  4.95 | top weight raw=0.99 | scaled=0.41
d= 768 | score spread= 19.86 | top weight raw=1.00 | scaled=0.45


In [6]:
def attention(X):
    s = X @ X.T / np.sqrt(X.shape[1])
    w = np.exp(s) / np.exp(s).sum(axis=1, keepdims=True)
    return w @ X

order = [2, 0, 1]                  # shuffle: sharply, profit, fell
X_shuf = X[order]

# Without positions: is "fell"'s output the same after shuffling?
print("No positions  -> same output for 'fell'?", np.allclose(attention(X)[1], attention(X_shuf)[2]))

# Add a seat number to each POSITION (sin/cos), then compare again
P = np.array([[np.sin(i), np.cos(i)] for i in range(3)])
print("With positions -> same output for 'fell'?", np.allclose(attention(X + P)[1], attention(X_shuf + P)[2]))

No positions  -> same output for 'fell'? True
With positions -> same output for 'fell'? False
